In [2]:
import os
import json
import numpy as np
import pandas as pd

from scipy.sparse import load_npz

# Path to processed datasets
data_path = "../data/processed"

# Load feature matrices
X_train = load_npz(os.path.join(data_path, "X_train.npz"))
X_val = load_npz(os.path.join(data_path, "X_val.npz"))
X_test = load_npz(os.path.join(data_path, "X_test.npz"))

# Load target labels
y_train = np.load(os.path.join(data_path, "y_train.npy"), allow_pickle=True)
y_val = np.load(os.path.join(data_path, "y_val.npy"), allow_pickle=True)
y_test = np.load(os.path.join(data_path, "y_test.npy"), allow_pickle=True)

# Load feature names
with open(os.path.join(data_path, "feature_names.json"), "r") as file:
    feature_names = json.load(file)

print("Datasets loaded successfully!")

print("\nTraining:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nTotal features:", len(feature_names))
print("Total disease classes:", len(np.unique(y_train)))

Datasets loaded successfully!

Training: (1025602, 975) (1025602,)
Validation: (132448, 975) (132448,)
Test: (134529, 975) (134529,)

Total features: 975
Total disease classes: 49


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Create baseline model
baseline_model = DummyClassifier(strategy="most_frequent")

# Train baseline
baseline_model.fit(X_train, y_train)

# Predict on validation data
baseline_predictions = baseline_model.predict(X_val)

# Evaluate
baseline_accuracy = accuracy_score(y_val, baseline_predictions)

baseline_precision = precision_score(
    y_val, baseline_predictions, average="weighted", zero_division=0
)

baseline_recall = recall_score(
    y_val, baseline_predictions, average="weighted", zero_division=0
)

baseline_f1_macro = f1_score(
    y_val, baseline_predictions, average="macro", zero_division=0
)

baseline_f1_weighted = f1_score(
    y_val, baseline_predictions, average="weighted", zero_division=0
)

print("BASELINE MODEL PERFORMANCE")
print("---------------------------")
print("Accuracy:", round(baseline_accuracy, 4))
print("Precision (Weighted):", round(baseline_precision, 4))
print("Recall (Weighted):", round(baseline_recall, 4))
print("F1 Score (Macro):", round(baseline_f1_macro, 4))
print("F1 Score (Weighted):", round(baseline_f1_weighted, 4))

BASELINE MODEL PERFORMANCE
---------------------------
Accuracy: 0.0654
Precision (Weighted): 0.0043
Recall (Weighted): 0.0654
F1 Score (Macro): 0.0025
F1 Score (Weighted): 0.008


In [4]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report
import time

# Create model
sgd_model = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    max_iter=20,
    tol=1e-3,
    random_state=42,
    n_jobs=-1
)

# Train model
start_time = time.time()

sgd_model.fit(X_train, y_train)

training_time = time.time() - start_time

print("SGD training completed!")
print("Training time:", round(training_time, 2), "seconds")

# Validation predictions
sgd_predictions = sgd_model.predict(X_val)

print("\nSGD Classifier Performance")
print("--------------------------")

SGD training completed!
Training time: 51.78 seconds

SGD Classifier Performance
--------------------------


In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy:",
      round(accuracy_score(y_val, sgd_predictions), 4))

print("Precision (Weighted):",
      round(precision_score(
          y_val, sgd_predictions,
          average="weighted", zero_division=0
      ), 4))

print("Recall (Weighted):",
      round(recall_score(
          y_val, sgd_predictions,
          average="weighted", zero_division=0
      ), 4))

print("F1 (Macro):",
      round(f1_score(
          y_val, sgd_predictions,
          average="macro", zero_division=0
      ), 4))

print("F1 (Weighted):",
      round(f1_score(
          y_val, sgd_predictions,
          average="weighted", zero_division=0
      ), 4))

Accuracy: 0.9961
Precision (Weighted): 0.9964
Recall (Weighted): 0.9961
F1 (Macro): 0.995
F1 (Weighted): 0.996


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

# Detailed performance for every disease
print(classification_report(
    y_val,
    sgd_predictions,
    labels=sgd_model.classes_,
    zero_division=0
))

# Check prediction distribution
print("\nNumber of diseases predicted:",
      len(np.unique(sgd_predictions)))

print("Incorrect predictions:",
      np.sum(y_val != sgd_predictions))

print("Correct predictions:",
      np.sum(y_val == sgd_predictions))

                                          precision    recall  f1-score   support

     Acute COPD exacerbation / infection       1.00      1.00      1.00      2076
                Acute dystonic reactions       1.00      1.00      1.00      3281
                        Acute laryngitis       0.99      1.00      1.00      3407
                      Acute otitis media       1.00      1.00      1.00      3474
                   Acute pulmonary edema       1.00      1.00      1.00      2500
                    Acute rhinosinusitis       1.00      0.81      0.90      1866
                      Allergic sinusitis       1.00      1.00      1.00      2136
                             Anaphylaxis       1.00      1.00      1.00      3754
                                  Anemia       1.00      1.00      1.00      6903
                     Atrial fibrillation       1.00      1.00      1.00      2609
                               Boerhaave       1.00      1.00      1.00      2075
               

In [7]:
report = classification_report(
    y_val,
    sgd_predictions,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

# Keep only disease rows
disease_report = report_df.loc[
    list(sgd_model.classes_),
    ["precision", "recall", "f1-score", "support"]
]

# Show 10 diseases with lowest recall
print(
    disease_report
    .sort_values("recall")
    .head(10)
    .round(4)
)

                         precision  recall  f1-score  support
Acute rhinosinusitis        1.0000  0.8114    0.8959   1866.0
Unstable angina             0.9996  0.9520    0.9752   2748.0
Croup                       1.0000  0.9963    0.9981    267.0
Viral pharyngitis           0.9990  0.9973    0.9982   8246.0
Acute laryngitis            0.9936  0.9977    0.9956   3407.0
Possible NSTEMI / STEMI     1.0000  0.9980    0.9990   2943.0
Bronchiectasis              1.0000  0.9996    0.9998   2319.0
Acute otitis media          1.0000  0.9997    0.9999   3474.0
Anaphylaxis                 1.0000  1.0000    1.0000   3754.0
Anemia                      1.0000  1.0000    1.0000   6903.0


In [8]:
# Find actual acute rhinosinusitis cases
mask = y_val == "Acute rhinosinusitis"

# Compare their actual and predicted labels
confusion_details = pd.DataFrame({
    "Actual": y_val[mask],
    "Predicted": sgd_predictions[mask]
})

print("Predictions for actual Acute rhinosinusitis cases:")
print(
    confusion_details["Predicted"]
    .value_counts()
    .head(10)
)

Predictions for actual Acute rhinosinusitis cases:
Predicted
Acute rhinosinusitis      1514
Chronic rhinosinusitis     352
Name: count, dtype: int64


In [10]:
from sklearn.svm import LinearSVC
import time

svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    dual="auto"
)

start_time = time.time()

svm_model.fit(X_train, y_train)  

svm_training_time = time.time() - start_time

print("LinearSVC training completed!")
print("Training time:", round(svm_training_time, 2), "seconds")

LinearSVC training completed!
Training time: 334.85 seconds


In [11]:
start_time = time.time()

svm_predictions = svm_model.predict(X_val)

svm_prediction_time = time.time() - start_time

print("Predictions generated successfully!")
print("Prediction time:",
      round(svm_prediction_time, 2), "seconds")

Predictions generated successfully!
Prediction time: 0.19 seconds


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

svm_accuracy = accuracy_score(y_val, svm_predictions)

svm_precision = precision_score(
    y_val, svm_predictions, average="weighted", zero_division=0
)

svm_recall = recall_score(
    y_val, svm_predictions, average="weighted", zero_division=0
)

svm_macro_f1 = f1_score(
    y_val, svm_predictions, average="macro", zero_division=0
)

svm_weighted_f1 = f1_score(
    y_val, svm_predictions, average="weighted", zero_division=0
)

print("LINEAR SVC - VALIDATION RESULTS")
print("-" * 40)
print(f"Accuracy:          {svm_accuracy:.4f}")
print(f"Weighted Precision:{svm_precision:.4f}")
print(f"Weighted Recall:   {svm_recall:.4f}")
print(f"Macro F1:          {svm_macro_f1:.4f}")
print(f"Weighted F1:       {svm_weighted_f1:.4f}")

LINEAR SVC - VALIDATION RESULTS
----------------------------------------
Accuracy:          0.9971
Weighted Precision:0.9973
Weighted Recall:   0.9971
Macro F1:          0.9966
Weighted F1:       0.9971


In [13]:
from sklearn.metrics import classification_report
import pandas as pd

svm_report = classification_report(
    y_val,
    svm_predictions,
    output_dict=True,
    zero_division=0
)

svm_report_df = pd.DataFrame(svm_report).T

print("10 diseases with the lowest recall:")
print(
    svm_report_df
    .drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")
    .sort_values("recall")
    [["precision", "recall", "f1-score", "support"]]
    .head(10)
    .round(4)
)

10 diseases with the lowest recall:
                         precision  recall  f1-score  support
Acute rhinosinusitis        0.9931  0.8446    0.9128   1866.0
Unstable angina             0.9989  0.9818    0.9903   2748.0
Chronic rhinosinusitis      0.9029  0.9960    0.9471   2717.0
Croup                       1.0000  0.9963    0.9981    267.0
Viral pharyngitis           0.9994  0.9979    0.9987   8246.0
Acute laryngitis            0.9950  0.9985    0.9968   3407.0
Possible NSTEMI / STEMI     1.0000  0.9986    0.9993   2943.0
URTI                        1.0000  0.9999    0.9999   8656.0
Anemia                      1.0000  1.0000    1.0000   6903.0
Atrial fibrillation         1.0000  1.0000    1.0000   2609.0


In [14]:
# Actual Acute rhinosinusitis cases
acute_mask = y_val == "Acute rhinosinusitis"

print("Actual Acute rhinosinusitis cases predicted as:")
print(
    pd.Series(svm_predictions[acute_mask])
    .value_counts()
    .head(5)
)

print("\nActual Chronic rhinosinusitis cases predicted as:")
chronic_mask = y_val == "Chronic rhinosinusitis"

print(
    pd.Series(svm_predictions[chronic_mask])
    .value_counts()
    .head(5)
)

Actual Acute rhinosinusitis cases predicted as:
Acute rhinosinusitis      1576
Chronic rhinosinusitis     290
Name: count, dtype: int64

Actual Chronic rhinosinusitis cases predicted as:
Chronic rhinosinusitis    2706
Acute rhinosinusitis        11
Name: count, dtype: int64


In [15]:
import os
import joblib

model_path = "../models"
os.makedirs(model_path, exist_ok=True)

joblib.dump(sgd_model, os.path.join(model_path, "sgd_model.pkl"))
joblib.dump(svm_model, os.path.join(model_path, "linear_svc_model.pkl"))

print("Both models saved successfully!")

Both models saved successfully!


In [16]:
import joblib
import os

model_path = "../models"

final_model = joblib.load(
    os.path.join(model_path, "linear_svc_model.pkl")
)

print("LinearSVC model loaded successfully!")

LinearSVC model loaded successfully!


In [17]:
import time

start_time = time.time()

test_predictions = final_model.predict(X_test)

test_prediction_time = time.time() - start_time

print("Test predictions generated successfully!")
print("Test dataset size:", len(y_test))
print("Prediction time:", round(test_prediction_time, 2), "seconds")

Test predictions generated successfully!
Test dataset size: 134529
Prediction time: 0.23 seconds


In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

test_accuracy = accuracy_score(y_test, test_predictions)

test_precision = precision_score(
    y_test, test_predictions,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    y_test, test_predictions,
    average="weighted",
    zero_division=0
)

test_macro_f1 = f1_score(
    y_test, test_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    y_test, test_predictions,
    average="weighted",
    zero_division=0
)

print("FINAL LINEARSVC - TEST RESULTS")
print("-" * 40)
print(f"Accuracy:           {test_accuracy:.4f}")
print(f"Weighted Precision: {test_precision:.4f}")
print(f"Weighted Recall:    {test_recall:.4f}")
print(f"Macro F1:           {test_macro_f1:.4f}")
print(f"Weighted F1:        {test_weighted_f1:.4f}")

FINAL LINEARSVC - TEST RESULTS
----------------------------------------
Accuracy:           0.9975
Weighted Precision: 0.9976
Weighted Recall:    0.9975
Macro F1:           0.9969
Weighted F1:        0.9974


In [19]:
from sklearn.metrics import classification_report
import pandas as pd

test_report = classification_report(
    y_test,
    test_predictions,
    output_dict=True,
    zero_division=0
)

test_report_df = pd.DataFrame(test_report).T

disease_report = test_report_df.drop(
    ["accuracy", "macro avg", "weighted avg"],
    errors="ignore"
)

print("10 diseases with lowest test recall:")
print(
    disease_report
    .sort_values("recall")
    [["precision", "recall", "f1-score", "support"]]
    .head(10)
    .round(4)
)

10 diseases with lowest test recall:
                          precision  recall  f1-score  support
Acute rhinosinusitis         0.9887  0.8611    0.9205   1829.0
Unstable angina              0.9989  0.9851    0.9920   2880.0
Chronic rhinosinusitis       0.9152  0.9934    0.9527   2747.0
Acute laryngitis             0.9944  0.9978    0.9961   3217.0
Viral pharyngitis            0.9992  0.9980    0.9986   8334.0
Possible NSTEMI / STEMI      1.0000  0.9990    0.9995   2911.0
Acute otitis media           1.0000  1.0000    1.0000   3516.0
Acute dystonic reactions     1.0000  1.0000    1.0000   3302.0
Anemia                       1.0000  1.0000    1.0000   6842.0
Atrial fibrillation          1.0000  1.0000    1.0000   2831.0


In [20]:
import json
import os

# Save detailed classification report
test_report_path = os.path.join(
    "../models", "test_classification_report.csv"
)

disease_report.to_csv(test_report_path)

# Save overall metrics
final_metrics = {
    "accuracy": float(test_accuracy),
    "weighted_precision": float(test_precision),
    "weighted_recall": float(test_recall),
    "macro_f1": float(test_macro_f1),
    "weighted_f1": float(test_weighted_f1),
    "test_samples": int(len(y_test)),
    "number_of_classes": int(len(disease_report))
}

metrics_path = os.path.join(
    "../models", "final_test_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(final_metrics, f, indent=4)

print("Evaluation reports saved successfully!")
print("1. test_classification_report.csv")
print("2. final_test_metrics.json")

Evaluation reports saved successfully!
1. test_classification_report.csv
2. final_test_metrics.json


In [21]:
import pandas as pd
import ast
import numpy as np
from scipy.sparse import load_npz

# Load one actual patient from the official test dataset
test_df = pd.read_csv(
    "../data/raw/ddxplus/release_test_patients.zip",
    compression="zip"
)

sample = test_df.iloc[0]

print("Age:", sample["AGE"])
print("Sex:", sample["SEX"])
print("Pathology:", sample["PATHOLOGY"])
print("Evidences:", sample["EVIDENCES"])

Age: 49
Sex: F
Pathology: GERD
Evidences: ['E_53', 'E_54_@_V_112', 'E_54_@_V_161', 'E_54_@_V_180', 'E_54_@_V_181', 'E_55_@_V_29', 'E_55_@_V_101', 'E_55_@_V_103', 'E_56_@_6', 'E_57_@_V_29', 'E_57_@_V_101', 'E_58_@_3', 'E_59_@_2', 'E_70', 'E_78', 'E_98', 'E_140', 'E_167', 'E_173', 'E_201', 'E_204_@_V_10', 'E_217']


In [22]:
import sys
sys.path.append("..")

from backend.feature_encoder import encode_patient
from scipy.sparse import load_npz

# Load the original preprocessed test matrix
X_test_saved = load_npz("../data/processed/X_test.npz")

# Encode the same actual patient using our backend encoder
actual_evidences = ast.literal_eval(sample["EVIDENCES"])

X_backend = encode_patient(
    sample["AGE"],
    sample["SEX"],
    actual_evidences
)

# Get the corresponding original feature vector
X_original = X_test_saved[0]

# Compare both vectors
difference = X_original - X_backend

print("Original shape:", X_original.shape)
print("Backend shape:", X_backend.shape)

print("Original non-zero features:", X_original.nnz)
print("Backend non-zero features:", X_backend.nnz)

print("Are both vectors identical:", difference.nnz == 0)
print("Number of differing values:", difference.nnz)

Original shape: (1, 975)
Backend shape: (1, 975)
Original non-zero features: 24
Backend non-zero features: 24
Are both vectors identical: True
Number of differing values: 0


In [23]:
import joblib

model = joblib.load("../models/linear_svc_model.pkl")

prediction = model.predict(X_backend)[0]

print("Actual pathology:", sample["PATHOLOGY"])
print("Predicted pathology:", prediction)
print("Prediction matches:", prediction == sample["PATHOLOGY"])

Actual pathology: GERD
Predicted pathology: GERD
Prediction matches: True
